In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import importlib
from config import *
from simulation.run_sim import Simulation
from simulation.aggregate_metrics import aggregate_metrics
from create_population.import_population import import_population


for pop_nr in range(1, 29):
    population_config = {
        "pop_file": "synthetic",
        "id": pop_nr
    }
    artificial_pop = import_population(**population_config)

    simulation = Simulation(
        population_ID=population_config["id"],
        population=artificial_pop,
        simulation_config=SIMULATION_SETTINGS,
    )

    simulation.run()
    simulation.export()


In [2]:
import pandas as pd
from pathlib import Path

# Folder containing the Excel files
folder = Path("../results")

# Find all Excel files starting with "results_"
files = sorted(folder.glob("results_*.xlsx"))

# Read and concatenate
metrics = pd.concat(
    (pd.read_excel(file, sheet_name="metrics") for file in files),
    ignore_index=True
)

print(f"Loaded {len(files)} files.")
print(metrics.head())

Loaded 108 files.
  method hv_selection       selection_type   bound_estimator  sample_size  \
0    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
1    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
2    MUS      nothing  systematic_sampling  Poisson_Stringer           30   
3    MUS      nothing  systematic_sampling  Poisson_Stringer           65   
4    MUS      nothing  systematic_sampling  Poisson_Stringer           65   

   confidence_level   z_score                        Population ID  \
0              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
1              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   
2              0.95  1.644854  BV_15pct_above_SI_F0.05_C0.1_R0.002   
3              0.80  0.841621  BV_15pct_above_SI_F0.05_C0.1_R0.002   
4              0.90  1.281552  BV_15pct_above_SI_F0.05_C0.1_R0.002   

   Population Book Value  Population Error Amount  ...  \
0           3.425979e+09             6.8

In [3]:
import re

# Population ID format (see main.py's _population_id()): BVBV_<N>pct_above_SI_F<f>_C<c>_R<r>
# e.g. "BVBV_5pct_above_SI_F0.2_C0.1_R0.01" -> BV_pop="5pct", f_target=0.2, corr_target=0.1, r_target=0.01
pattern = r"^BVBV_(?P<bv>\d+pct)_above_SI_F(?P<f>[\d.]+)_C(?P<c>[\d.]+)_R(?P<r>[\d.]+)$"
extracted = metrics["Population ID"].astype(str).str.extract(pattern)

metrics["BV_pop"] = extracted["bv"]
metrics["f_target"] = extracted["f"].astype(float)
metrics["corr_target"] = extracted["c"].astype(float)
metrics["r_target"] = extracted["r"].astype(float)

path = RESULTS_DIR / "main_simulation_results.xlsx"
with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        metrics.to_excel(writer, sheet_name="metrics", index=False)
        for group_col, table in aggregate_metrics(metrics, "main").items():
            table.to_excel(writer, sheet_name=f"agg_{group_col}"[:31], index=True)

In [4]:
for file in files:
    results = pd.read_excel(file, sheet_name="results (€)")
    if (results["SE_pred"]<0).any():
        print(f"Negative SE_pred found in {file.name}")

Negative SE_pred found in results_BV_15pct_above_SI_F0.05_C0.5_R0.03.xlsx
Negative SE_pred found in results_BV_15pct_above_SI_F0.05_C0.5_R0.04.xlsx
Negative SE_pred found in results_BV_15pct_above_SI_F0.2_C0.5_R0.03.xlsx
Negative SE_pred found in results_BV_15pct_above_SI_F0.2_C0.5_R0.04.xlsx


KeyboardInterrupt: 

In [ ]:
results.head()

In [ ]:
results["SE_pred"]

### Look at specific cases

In [33]:
metrics[metrics['Population ID']==3]

,method,hv_selection,selection_type,bound_estimator,sample_size,confidence_level,z_score,Population ID,Population Book Value,Population Error Amount,...,r_target,Correct Acceptance,Incorrect Rejection,Incorrect Acceptance,Correct Rejection,Relative Bias of Error Estimation,Relative Precision of Error Estimation,Precision of Error Estimation in %,Relative Bias of Precision Estimation,Relative Precision of Precision Estimation


In [34]:
metrics[metrics['Population Error Amount'] >= metrics['Population Book Value']*0.02]['Population ID'].unique()

<StringArray>
[ 'BVBV_15pct_above_SI_F0.05_C0.1_R0.025',
   'BVBV_15pct_above_SI_F0.05_C0.1_R0.03',
   'BVBV_15pct_above_SI_F0.05_C0.1_R0.05',
 'BVBV_15pct_above_SI_F0.05_C0.25_R0.025',
  'BVBV_15pct_above_SI_F0.05_C0.25_R0.03',
  'BVBV_15pct_above_SI_F0.05_C0.25_R0.05',
  'BVBV_15pct_above_SI_F0.05_C0.5_R0.025',
   'BVBV_15pct_above_SI_F0.05_C0.5_R0.03',
   'BVBV_15pct_above_SI_F0.05_C0.5_R0.05',
   'BVBV_15pct_above_SI_F0.2_C0.1_R0.025',
    'BVBV_15pct_above_SI_F0.2_C0.1_R0.03',
    'BVBV_15pct_above_SI_F0.2_C0.1_R0.05',
  'BVBV_15pct_above_SI_F0.2_C0.25_R0.025',
   'BVBV_15pct_above_SI_F0.2_C0.25_R0.03',
   'BVBV_15pct_above_SI_F0.2_C0.25_R0.05',
   'BVBV_15pct_above_SI_F0.2_C0.5_R0.025',
    'BVBV_15pct_above_SI_F0.2_C0.5_R0.03',
    'BVBV_15pct_above_SI_F0.2_C0.5_R0.05',
   'BVBV_15pct_above_SI_F0.5_C0.1_R0.025',
    'BVBV_15pct_above_SI_F0.5_C0.1_R0.03',
    'BVBV_15pct_above_SI_F0.5_C0.1_R0.05',
  'BVBV_15pct_above_SI_F0.5_C0.25_R0.025',
   'BVBV_15pct_above_SI_F0.5_C0.25_R0.03

In [35]:
populations_above_mat = [14, 15, 16, 17, 18, 19, 20, 21, 22, 1]
metrics_filtered = metrics[~metrics['Population ID'].isin(populations_above_mat)]
metrics_filtered.head()

,method,hv_selection,selection_type,bound_estimator,sample_size,confidence_level,z_score,Population ID,Population Book Value,Population Error Amount,...,r_target,Correct Acceptance,Incorrect Rejection,Incorrect Acceptance,Correct Rejection,Relative Bias of Error Estimation,Relative Precision of Error Estimation,Precision of Error Estimation in %,Relative Bias of Precision Estimation,Relative Precision of Precision Estimation
0,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.80,0.841621,BVBV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,0.002,0.0,0.0570,NaN,NaN,-0.002919,3.369538,0.006739,0.876144,0.047078
1,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.90,1.281552,BVBV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,0.002,0.0,0.0560,NaN,NaN,-0.002707,5.209974,0.010420,0.866259,0.077035
2,MUS,nothing,systematic_sampling,Poisson_Stringer,30,0.95,1.644854,BVBV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,0.002,0.0,0.0554,NaN,NaN,-0.023358,6.593149,0.013186,0.869839,0.095586
3,MUS,nothing,systematic_sampling,Poisson_Stringer,65,0.80,0.841621,BVBV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,0.002,0.0,0.0069,NaN,NaN,-0.022691,2.278685,0.004557,0.821154,0.066793
4,MUS,nothing,systematic_sampling,Poisson_Stringer,65,0.90,1.281552,BVBV_15pct_above_SI_F0.05_C0.1_R0.002,3.425979e+09,6.851959e+06,...,0.002,0.0,0.0085,NaN,NaN,0.047030,3.619381,0.007239,0.802243,0.112072


In [36]:
group_cols = ["method", "sample_size", "hv_selection", "selection_type", "bound_estimator"]
excluded_cols = group_cols
value_cols = metrics_filtered.columns.difference(excluded_cols)
summary = (
        metrics_filtered
        .groupby(group_cols)["Coverage"]
        .describe()
        .T
    )

In [37]:
summary

method                          MUS                                  \
sample_size                     30                                    
hv_selection              iterative             nothing               
selection_type  systematic_sampling systematic_sampling               
bound_estimator                  HH   Binomial_Stringer      Moment   
count                    360.000000               360.0  360.000000   
mean                       0.639654                 1.0    0.991427   
std                        0.183227                 0.0    0.041074   
min                        0.169300                 1.0    0.693000   
25%                        0.585525                 1.0    1.000000   
50%                        0.653700                 1.0    1.000000   
75%                        0.766800                 1.0    1.000000   
max                        1.000000                 1.0    1.000000   

method                                                                    \
sample_size                                      65                        
hv_selection                               iterative             nothing   
selection_type                   systematic_sampling systematic_sampling   
bound_estimator Poisson_Stringer                  HH   Binomial_Stringer   
count                      360.0          360.000000          360.000000   
mean                         1.0            0.839632            0.988546   
std                          0.0            0.142775            0.036197   
min                          1.0            0.566400            0.802600   
25%                          1.0            0.723025            1.000000   
50%                          1.0            0.823450            1.000000   
75%                          1.0            1.000000            1.000000   
max                          1.0            1.000000            1.000000   

method                                                            \
sample_size                                                  100   
hv_selection                                           iterative   
selection_type                               systematic_sampling   
bound_estimator      Moment Poisson_Stringer                  HH   
count            360.000000       360.000000          360.000000   
mean               0.982899         0.987754            0.822849   
std                0.052395         0.037384            0.119813   
min                0.624200         0.821200            0.574000   
25%                1.000000         1.000000            0.731475   
50%                1.000000         1.000000            0.812850   
75%                1.000000         1.000000            0.893125   
max                1.000000         1.000000            1.000000   

method                                                            \
sample_size                                                        
hv_selection                nothing                                
selection_type  systematic_sampling                                
bound_estimator   Binomial_Stringer      Moment Poisson_Stringer   
count                    360.000000  360.000000       360.000000   
mean                       0.986492    0.968539         0.981895   
std                        0.026996    0.069763         0.036273   
min                        0.852200    0.573900         0.816800   
25%                        0.993175    0.964450         0.987975   
50%                        1.000000    1.000000         1.000000   
75%                        1.000000    1.000000         1.000000   
max                        1.000000    1.000000         1.000000   

method                                                               \
sample_size                     150                                   
hv_selection              iterative             nothing               
selection_type  systematic_sampling systematic_sampling               
bound_estimator                  